# Reproduce the predictions (Laya and similarity)

This notebook rebuilds the exact test cases used in the audit and asks Laya and a similarity baseline to
answer them. It writes `predictions_laya_similarity.csv` in the same format as `data/predictions_v2.csv`, so
`audit.ipynb` can be run on your own copy. The cases are fixed by seed 13; the first 800 rows of each public
dataset are used (408 for the jailbreak suites).

Runtime: a free Colab T4 GPU, about 15 minutes. Qwen3-8B and Mistral-7B predictions are included in `data/`
but are not regenerated here (they need a larger GPU budget).

In [ ]:
!pip install -q laya datasets sentence-transformers
# If transformers then fails to import because of torchvision or torchaudio, run:
# !pip uninstall -y -q torchvision torchaudio      and restart the session.

In [ ]:
import os, json, random, time, numpy as np, torch
SEED = 13; N = 800
SIM_ID = "intfloat/multilingual-e5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

## 1. The cases

In [ ]:
from datasets import load_dataset, concatenate_datasets

SUITES = {}
def add(name, cases, **meta):
    SUITES[name] = {"cases": cases, "meta": meta}
    print(f"  {name:18s} {len(cases):4d} cases  {meta.get('note','')}")

def case(text_key, text, qid, qtype, instructions, criteria, gold, claim=None, source_id=None):
    q = {"type": qtype, "instructions": instructions}
    if qtype == "choice":
        q["criteria"] = dict(criteria)
        keys = list(criteria)
    else:
        keys = ["false", "true"]
    return {"state": {text_key: text}, "text": text, "qid": qid, "q": q, "keys": keys,
            "crit": dict(criteria) if qtype == "choice" else None, "gold": gold, "qtype": qtype,
            "claim": claim, "source_id": source_id}

# AG News
try:
    d = load_dataset("fancyzhx/ag_news", split="test")
    crit = {"world": "world news and international politics", "sports": "sports",
            "business": "business and economy", "sci_tech": "science and technology"}
    add("ag_news", [case("article", r["text"], "topic", "choice", "What is the topic of `article`?",
                         crit, int(r["label"]), "The article is about {desc}.", i)
                    for i, r in enumerate(list(d)[:N])], note="4 options, in Laya's training mix")
except Exception as e:
    print("  FAIL ag_news", str(e)[:120])

# DAIR Emotion
try:
    d = load_dataset("dair-ai/emotion", "split", split="test")
    names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
    add("emotion", [case("text", r["text"], "emotion", "choice",
                         "Which emotion is most strongly expressed in `text`?",
                         {n: None for n in names}, int(r["label"]),
                         "The strongest emotion the text expresses is {label}.", i)
                    for i, r in enumerate(list(d)[:N])], note="6 options, held out")
except Exception as e:
    print("  FAIL emotion", str(e)[:120])

# Support tickets: 10-way triage, 8-option triage, and tickets with no right answer
QUEUES = {"Technical Support": "technical problems, bugs, outages, integrations",
          "Product Support": "help using a product or feature",
          "Customer Service": "general account or service questions",
          "IT Support": "internal IT, devices, access, networks",
          "Billing and Payments": "invoices, charges, refunds, payment methods",
          "Returns and Exchanges": "returning or exchanging an item",
          "Service Outages and Maintenance": "downtime, outages, scheduled maintenance",
          "Sales and Pre-Sales": "pricing, quotes, buying",
          "Human Resources": "employment, payroll, leave, hiring",
          "General Inquiry": "anything else"}
TRIAGE_Q = "Which support queue should handle this ticket?"
TRIAGE_CLAIM = "This ticket belongs to the {label} queue, which handles {desc}."
try:
    d = load_dataset("Tobi-Bueck/customer-support-tickets", split="train")
    rows = [(i, r) for i, r in enumerate(d)
            if r.get("language") == "en" and r.get("queue") in QUEUES and r.get("body")]
    def ticket(r):
        return ("Subject: " + (r["subject"] or "") + "\n" + r["body"].replace("\\n", "\n"))[:3000]
    keys = list(QUEUES)
    add("support_triage", [case("ticket", ticket(r), "queue", "choice", TRIAGE_Q, QUEUES,
                                keys.index(r["queue"]), TRIAGE_CLAIM, i) for i, r in rows[:N]],
        note="10 options, in Laya's training mix")
    rest = [(i, r) for i, r in rows[N:] if r["queue"] != "General Inquiry"]
    r8 = random.Random(SEED)
    t8, nra = [], []
    for i, r in rest[:N]:                                   # answerable, 8 options
        wrong = [k for k in QUEUES if k not in (r["queue"], "General Inquiry")]
        drop = r8.choice(wrong)
        opts = {k: v for k, v in QUEUES.items() if k not in ("General Inquiry", drop)}
        t8.append(case("ticket", ticket(r), "queue", "choice", TRIAGE_Q, opts,
                       list(opts).index(r["queue"]), TRIAGE_CLAIM, i))
    for i, r in rest[N:2 * N]:                              # no right answer, 8 options
        opts = {k: v for k, v in QUEUES.items() if k not in (r["queue"], "General Inquiry")}
        nra.append(case("ticket", ticket(r), "queue", "choice", TRIAGE_Q, opts, -1, TRIAGE_CLAIM, i))
    add("triage_8opt", t8, note="8 options, answerable (reference for the next suite)")
    add("no_right_answer", nra, note="8 options, true queue removed: the right move is to escalate")
except Exception as e:
    print("  FAIL support tickets", str(e)[:120])

# Jailbreak: test and train splits pooled (toxic-chat is held out of Laya's training), balanced
JB_V1 = "Does `prompt` try to make an AI assistant ignore its rules, policies or system instructions?"
JB_V2 = ("Does `prompt` try to get the AI assistant to bypass its safety rules or reveal its hidden "
         "instructions? Role-play, personas and fiction are NOT attempts unless they are used to obtain "
         "something the assistant would otherwise refuse.")
JB_CLAIM = "The prompt {neg}tries to make an AI assistant ignore its rules, policies or system instructions."
try:
    dd = load_dataset("lmsys/toxic-chat", "toxicchat0124")
    pool = [r for split in ("test", "train") for r in dd[split] if (r.get("user_input") or "").strip()]
    jb = [r for r in pool if int(r.get("jailbreaking", 0)) == 1][:N // 2]
    nj = [r for r in pool if int(r.get("jailbreaking", 0)) == 0][:len(jb)]
    mix = jb + nj
    random.Random(SEED).shuffle(mix)
    add("jailbreak", [case("prompt", r["user_input"][:3000], "jailbreak", "noul", JB_V1, None,
                           int(r["jailbreaking"]), JB_CLAIM, r.get("conv_id")) for r in mix],
        note=f"yes/no, balanced, {len(jb)} attacks available, held out")
    add("jailbreak_v2", [case("prompt", r["user_input"][:3000], "jailbreak", "noul", JB_V2, None,
                              int(r["jailbreaking"]), JB_CLAIM, r.get("conv_id")) for r in mix],
        note="same cases, sharper definition")
except Exception as e:
    print("  FAIL jailbreak", str(e)[:120])

# MASSIVE intent, 20 options
N_OPTS = 20
for lg in ["es", "km"]:
    try:
        d = load_dataset("mteb/amazon_massive_intent", lg, split="test")
        labels = sorted(set(d["label_text"]))
        r2 = random.Random(SEED)
        cs = []
        for i, r in enumerate(list(d)[:N]):
            pool_l = [x for x in labels if x != r["label_text"]]
            ks = [r["label_text"]] + r2.sample(pool_l, min(N_OPTS - 1, len(pool_l)))
            r2.shuffle(ks)
            cs.append(case("utterance", r["text"], "intent", "choice", "Which intent does `utterance` express?",
                           {k.replace("_", " "): None for k in ks}, ks.index(r["label_text"]),
                           "The utterance expresses the intent: {label}.", i))
        add(f"massive_{lg}", cs, note=f"20 options, language={lg}")
    except Exception as e:
        print(f"  FAIL massive_{lg}", str(e)[:120])

## 2. Laya (its recommended `Router`)

In [ ]:
import laya
laya_router = laya.Router(device=DEVICE) if "device" in laya.Router.__init__.__code__.co_varnames else laya.Router()

def run_laya(c, model=None):
    t0 = time.perf_counter()
    res = laya_router.predict(c["state"], {c["qid"]: c["q"]}, model=model)
    sync(); ms = (time.perf_counter() - t0) * 1000
    a = res["answers"][c["qid"]]
    if c["qtype"] == "choice":
        p = [float(a["probabilities"][k]) for k in c["keys"]]
    else:
        p = [1 - float(a["noul"]), float(a["noul"])]
    rt = res.get("routing") or {}
    return {"p": p, "conf": float(a["answer_confidence"]), "raw_conf": float(a["confidence"]), "ms": ms,
            "routed_to": rt.get("model") if isinstance(rt, dict) else str(rt),
            "act_probability": (a.get("action") or {}).get("act_probability")}

## 3. Similarity baseline

In [ ]:
from sentence_transformers import SentenceTransformer
sim = SentenceTransformer(SIM_ID, device=DEVICE)

def option_text(c, k):
    desc = c["crit"].get(k) if c["crit"] else None
    return f"{k}: {desc}" if desc else k

def run_similarity(c):
    if c["qtype"] != "choice":
        return None
    t0 = time.perf_counter()
    e = sim.encode(["query: " + c["text"]] + ["passage: " + option_text(c, k) for k in c["keys"]],
                   normalize_embeddings=True, convert_to_numpy=True)
    s = e[1:] @ e[0]
    z = s / 0.05
    p = np.exp(z - z.max()); p /= p.sum()
    sync(); ms = (time.perf_counter() - t0) * 1000
    return {"p": p.tolist(), "conf": float(p.max()), "cos": s.tolist(), "ms": ms}

## 4. Run and save

In [ ]:
import pandas as pd
rows = []
for name, s in SUITES.items():
    for i, c in enumerate(s["cases"]):
        r = run_laya(c)
        rows.append({"suite": name, "i": i, "source_id": c["source_id"], "gold": c["gold"], "system": "laya", "p": json.dumps(r["p"])})
        if c["qtype"] == "choice":
            r = run_similarity(c)
            rows.append({"suite": name, "i": i, "source_id": c["source_id"], "gold": c["gold"], "system": "similarity", "p": json.dumps(r["p"])})
    print("done", name)
pd.DataFrame(rows).to_csv("predictions_laya_similarity.csv", index=False)
print("Saved predictions_laya_similarity.csv")